In [1]:
import pyGinkgo as pg
import numpy as np

### Direct solver bindings

In [2]:
fn = 'm1.mtx'
dev = pg.device("cuda")
mtx = pg.read(path=fn, dtype="double", format="Csr")
n_rows = mtx.size[0]

b = pg.as_tensor(
  device=dev, dim=(n_rows,1), dtype="double", fill=1.0
)
x = pg.as_tensor(
  device=dev, dim=(n_rows,1), dtype="double", fill=0.0
)

# Create ILU preconditioner
preconditioner = pg.preconditioner.Ilu(dev, mtx)

# Setup GMRES solver
solver = pg.solver.gmres(dev, mtx, preconditioner=preconditioner,
    max_iters=1000, krylov_dim=30, reduction_factor=1e-06
)

#Apply
logger, result = solver.apply(b, x)

In [3]:
result_cpu = result.copy_to_host()

/home/ubuntu/projects/work/cpp/pyGinkgo/src/cpp_bindings/dense.cppWarning creating a copy of dense


In [4]:
for i in range(10):
    print(result_cpu.at(i, 0), end=", ")

0.6044527378678369, 0.8250277658139552, 0.9154476498993512, 0.9545267011460608, 0.9719330030000276, 0.9798380401271495, 0.983477083786615, 0.9851691223649147, 0.9859619246219566, 0.9863356545776443, 

### Config solver

In [5]:
args = {
    "type": "solver::Gmres",
    "krylov_dim": 30,
    "preconditioner": {
        "type": "preconditioner::Jacobi",
        "max_block_size": 1
    },
    "criteria": [
        {"type": "Iteration", "max_iters": 1000},
        {
          "type": "ResidualNorm", 
          "reduction_factor": 1e-6, 
          "baseline": "rhs_norm"
        }
    ],
}

In [6]:
b = pg.as_tensor(
  device=dev, dim=(n_rows,1), dtype="double", fill=1.0
)
x = pg.as_tensor(
  device=dev, dim=(n_rows,1), dtype="double", fill=0.0
)

solver = pg.generate_solver(mtx, args)

logger, result = solver.apply(b, x)

In [7]:
result_cpu_2 = result.copy_to_host()

/home/ubuntu/projects/work/cpp/pyGinkgo/src/cpp_bindings/dense.cppWarning creating a copy of dense


### Checking difference between two solvers

In [8]:
result_cpu_2.add_scaled(
    pg.as_tensor(device=dev, dim=(1, 1), fill=-1),
    result_cpu
)

In [9]:
for i in range(10):
    print(result_cpu_2.at(i, 0), end=", ")

2.3617819921994965e-09, -5.443318129572106e-09, -2.8040688659913826e-09, -1.7958602271939128e-08, -4.790551955125011e-09, -2.7632607979555246e-08, -5.421204707367622e-09, -3.6818322501908085e-08, -1.0452056553589273e-08, -5.7313320867002915e-08, 